# L1: Text classification

Text classification is the task of sorting text documents into predefined classes. In this lab, we will be working with speeches held in the Swedish parliament, originally provided by [Riksdagens öppna data](http://data.riksdagen.se/). The classification task is to predict the speaker’s affiliation to a political block (left-wing/right-wing) based on the speech they gave.

Along the way, we will see examples for how to use different Python libraries, such as:

- [pandas](https://pandas.pydata.org/) for loading the data
- [scikit-learn](https://scikit-learn.org/) for preprocessing the data and training a classifier

Don’t worry if you are not familiar with these tools yet – we will show examples for how to use them and explain what they do, but we also provide links to their documentation in case you want to learn more or use them for your own projects.

The following line loads some helper functions that are used in parts of the notebook:

In [30]:
from liu_lt import check_solutions, find_file

## Loading the data

The data used in this lab consists of speeches held in the Swedish parliament in the 2023/24 and 2024/25 sessions. We preprocessed the raw data from [Riksdagens open data](http://data.riksdagen.se/) to produce a collection of speeches and their associated party labels. Speeches are divided into two files, in a compressed JSON format:

* `anforande-202324.json.bz2` (speeches from the 2023/24 parliamentary session)
* `anforande-202425.json.bz2` (speeches from the 2024/25 parliamentary session)

First, let's load the speeches into a Pandas DataFrame:

In [31]:
import bz2
import pandas as pd

with bz2.open(find_file("anforande-202324.json.bz2")) as f:
    speeches_202324 = pd.read_json(f)

with bz2.open(find_file("anforande-202425.json.bz2")) as f:
    speeches_202425 = pd.read_json(f)

If we look at the speeches from the 2023/24 session, we can see that the data contains an ID, the text of the speech, a party label, and the political block ("wing", either L=left-wing or R=right-wing):

In [32]:
speeches_202324

,id,text,party,wing
0,HB-002-004,Eders Majestäter! Eders Kungliga Högheter! Her...,M,R
1,HB-003-001,Fru talman! Gårdagen var en firandets dag i ri...,S,L
2,HB-003-002,Fru talman! Jag yrkar bifall till Vänsterparti...,V,L
3,HB-003-003,Fru talman! Vi debatterar regeringens proposit...,SD,R
4,HB-003-004,Fru talman! Sverige är i ett nytt läge. Det är...,C,R
...,...,...,...,...
15190,HB-141-045,Fru talman! Jag är inte helt säker på att jag ...,M,R
15191,HB-141-046,Fru talman! Jag blir lite förvånad över att st...,S,L
15192,HB-141-047,Fru talman! Detta är onekligen ett allvarligt ...,M,R
15193,HB-141-048,Fru talman! Fint att jag ändå har två minuter ...,S,L


We can index this DataFrame to get specific columns or rows from it.  For example, the see the full text of the speech on row 42, we can do:

In [33]:
speeches_202324["text"][42]

'Herr talman! Regeringen har presenterat stora utgifter, i storleksordningen 20 miljarder kronor. Det handlar endast om skattesänkningar som går mest till dem som är rikast i Sverige. Varför det vore en bättre prioritering av statens medel förstår i varje fall inte de moderata regionråden. Här svarar inte Ulf Kristersson på frågan. Varför är detta mer prioriterat än att rädda dem som är närmast verksamheten och ser vad det är som händer? Vi är i en kris. Då måste vi väl prioritera vården, skolan och omsorgen. Ulf Kristersson kan inte på allvar mena att han tror att människor som behåller jobbet i skolan är inflationsdrivande. Det är i så fall mer inflationsdrivande att sänka skatter. Det måste finnas ett svar här. Dina regionråd är väldigt tydliga. De säger att de kan behöva lägga ned stora verksamheter. Vad är Ulf Kristerssons besked till de egna regionråden? (Applåder)'

Here is the political affiliation of the speaker in this example:

In [34]:
speeches_202324["wing"][42]

'L'

In the following, we want to train a classifier with this data as follows:

- As our **training data**, we use the speeches from the 2023/24 parliamentary session.
- As our **test data**, we use the speeches from the 2024/25 parliamentary session.
- Each row is one data sample, with the **"text" column** being the input to the classifier, and the **"wing" column** being the class label that we want to predict.

## Training a classifier

The following cell shows how you can use scikit-learn to build a classification pipeline.  It combines a bag-of-words vectorizer and a Naive Bayes classifier, and trains them on the training data from our parliament speeches:

In [35]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Create a pipeline with a vectorizer + a classifier
vectorizer = CountVectorizer()
classifier = MultinomialNB()
model = Pipeline([
    ('vec', vectorizer),
    ('clf', classifier)
])

# Train (or "fit", in scikit-learn terminology) the classifier on the training dataset
model.fit(speeches_202324['text'], speeches_202324['wing'])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('vec', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


We can call `model.predict()` on some list of input texts to make predictions. Let's see if the model correctly predicts the example we looked at above (from the training data):

In [36]:
model.predict([speeches_202324["text"][42]])

array(['L'], dtype='<U1')

In the rest of this lab, we will try to better understand what is happening here by inspecting what the “CountVectorizer” does, properly evaluating this classifier on test data, and comparing it against a baseline.

## Task 1: Inspecting the CountVectorizer

The [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html), as the name suggests, converts a text into vectors by counting tokens.  This is also sometimes called a “bag-of-words” representation of the text. You should recall from the lectures that this always requires us to _tokenize_ the text first, and potentially perform some other _preprocessing_ steps.  The CountVectorizer in scikit-learn actually does a lot of these things by default, so let’s take a closer look at what is actually happening here.

Let’s take a short example document from the training data and transform it into a vector with the CountVectorizer from our pipeline:

In [37]:
example = speeches_202324["text"][528]  # We can use any other document, but this one here is rather short.
print(example)
print()

vector = vectorizer.transform([example])
print(vector)

Fru talman! Jag är väldigt nöjd med att ledamöterna här inne, i synnerhet Martin Melin, är nöjda med svaren. Det är jag nöjd med. (Applåder)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 18 stored elements and shape (1, 86569)>
  Coords	Values
  (0, 2951)	1
  (0, 4137)	1
  (0, 12896)	1
  (0, 20687)	1
  (0, 29437)	1
  (0, 31765)	1
  (0, 33285)	2
  (0, 41412)	1
  (0, 44849)	1
  (0, 45247)	3
  (0, 45715)	1
  (0, 50881)	2
  (0, 50882)	1
  (0, 70483)	1
  (0, 71025)	1
  (0, 72008)	1
  (0, 82346)	1
  (0, 84583)	3


The output here can be a bit hard to understand at first.  Remember that for each token that we want to include in our bag-of-words vector, there must be a dimension in the vector that corresponds to this token.  This means that in practice, we have a very large vector where most of the dimensions are zero.  The output above skips all the dimensions that are zero, and only shows the ones where the vector has a non-zero value.  If you see a row like the following:

```
(0, 20687)	1
```

That means that the vector has the value 1 in the 20687-th dimension.  **But what does that dimension represent?**  We can find out by inspecting the vectorizer's vocabulary.  To make this easier, we have prepared a function that does this for you – you don’t need to understand how exactly it works, you just need to use it to inspect the example document:

In [38]:
def show_vector_with_features(vectorizer, text):
    """Transforms the `text` with `vectorizer` and shows each non-zero dimension
    together with the feature (= token) that it represents."""
    vector = vectorizer.transform([text])
    feats = vectorizer.get_feature_names_out()
    entries = []
    for idx in vector.nonzero()[1]:
        entries.append((int(idx), feats[idx], int(vector[0, idx])))
    return pd.DataFrame.from_records(entries, columns=["dimension", "feature", "value"], index="dimension")

In [39]:
# Run this function on the example document
show_vector_with_features(vectorizer, example)

,feature,value
dimension,,
2951,applåder,1
4137,att,1
12896,det,1
20687,fru,1
29437,här,1
31765,inne,1
33285,jag,2
41412,ledamöterna,1
44849,martin,1


If you compare this table to the output you got above (where we just did `print(vector)`), you should see that the numbers under "dimension" and "value" are exactly the same, but now there is also a "feature" column that shows you what token the dimension corresponds to.  Compare what you see to the original text, then write down your observations below.

<div class="alert alert-info">
<strong>What preprocessing steps you know from the lecture (<i>e.g.</i> tokenization, lowercasing, lemmatization, ...) has the CountVectorizer done?</strong>  Write a brief answer with your observations below and give concrete examples from the document above.
</div>

The CountVectorizer automatically performs tokenization and lowercasing. 

All words in the document are converted to lowercase, so for example “Sverige” and “sverige” are treated the same. It also removes punctuation and splits the text into word tokens based on whitespace and punctuation. 

No lemmatization or stemming is performed, so different word forms such as “arbeta” and “arbetar” are treated as separate features. 

Stopwords can be removed if specified, but it is not enabled by default.

## Task 2: Evaluate the classifier

Let's now evaluate our classifier on the test data.  Scikit-learn conveniently provides a function called [`classification_report()`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) that returns all kinds of evaluation metrics, including accuracy, precision, recall, and F1-score.  The function should be called with two arguments:

- The _first_ argument is the **true labels**, i.e. the gold-standard annotations from the test data.
- The _second_ argument is the **predicted labels**, i.e. what the classifier has predicted for the texts in the test data.  To get the classifier's predictions, you can call `model.predict()` with the texts from the test data as an argument.

The `classification_report()` function then returns a string with the results. To make the output look as intended, you also need to `print()` the string.

<div class="alert alert-info">
    <strong>Add code to evaluate the classifier on the test data and print a classification report!</strong>  <i>(Remember that the test data is all speeches from the 2024/25 session.)</i>
</div>

In [40]:
from sklearn.metrics import classification_report

# YOUR CODE HERE
# Evaluate the classifier on the test data (speeches from 2024/25)
y_true = speeches_202425['wing']
y_pred = model.predict(speeches_202425['text'])
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           L       0.79      0.76      0.78      5455
           R       0.85      0.87      0.86      8594

    accuracy                           0.83     14049
   macro avg       0.82      0.82      0.82     14049
weighted avg       0.83      0.83      0.83     14049



If your solution is correct, you should see some numbers in tabular form, with an accuracy score around 0.83 (= 83%).

## Task 3: Re-implement the evaluation functions

From the lectures, we know how to compute accuracy, precision, recall, and F1-score.  Your next task is to test your understanding of these metrics by doing your own implementation of these functions “by hand”, i.e. without relying on any code from scikit-learn or similar libraries.

<div class="alert alert-info">
    <strong>Write code for each of the evaluation functions in the cells below!</strong>  Some hints:

  - If you need to divide by a number, make sure to check that number is not zero.  In each function, if you would divide by zero, the function should return zero instead.
  - Python's [zip function](https://docs.python.org/3/library/functions.html#zip) can be very useful here to iterate over two lists simultaneously.
  - At the end of this section, there is code for you to test if your implementation appears to be correct.
</div>

<div class="alert alert-warning">
⚠️ You <strong>may not</strong> use a function from a library (like scikit-learn) to compute the scores; you must compute them yourselves by comparing the true and predicted labels.
</div>

### Accuracy

This function takes a list of gold-standard labels (`true`) and a list of predicted labels (`pred`) and returns the accuracy as a floating-point number between 0 and 1.

In [41]:
def accuracy(true, pred):
    # YOUR CODE HERE
    """Compute the accuracy of a classifier by comparing the true labels and predicted labels."""
    true = list(true)
    pred = list(pred)
    if len(true) == 0:
        return 0.0
    correct = sum(1 for t, p in zip(true, pred) if t == p)
    return correct / len(true)

### Precision

This function takes a list of gold-standard labels (`true`), a list of predicted labels (`pred`), and a target class (`target`), and returns the classifier’s precision at predicting documents with class `target` as a floating-point number between 0 and 1.

In [42]:
def precision(true, pred, target):
    # YOUR CODE HERE
    """Compute the class-specific precision of a classifier by comparing the true labels and predicted labels."""
    true = list(true)
    pred = list(pred)


    # true positives
    tp = sum(1 for t, p in zip(true, pred) if p == target and t == target)
    
    # predicted positives:
    predicted_pos = sum(1 for p in pred if p == target)
    if predicted_pos == 0:
        return 0.0
    return tp / predicted_pos

### Recall

This function works the same as the previous one, but instead computes the recall.

In [43]:
def recall(true, pred, target):
    # YOUR CODE HERE
    
    """Compute the class-specific recall of a classifier by comparing the true labels and predicted labels."""
    true = list(true)
    pred = list(pred)

    # true
    tp = sum(1 for t, p in zip(true, pred) if p == target and t == target)
    
    # actual
    actual_pos = sum(1 for t in true if t == target)
    if actual_pos == 0:
        return 0.0
    return tp / actual_pos

### F1-score

This function takes the values for precision and recall (as two floating-point numbers between 0 and 1), and returns the F1-score (= the harmonic mean of the two).

In [44]:
def f1_score(precision, recall):
    # YOUR CODE HERE
    
    """Compute the F1-score from precision and recall."""
    if (precision + recall) == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

### 🤞 Test your code

Before you continue, you can test your implementation by running the following cell, which will compute some evaluation scores on a “toy” dataset and check if your functions return the expected values:

In [45]:
toy_data = pd.DataFrame.from_dict({
    "true": ["A", "A", "B", "B", "C"],
    "pred": ["A", "A", "A", "B", "B"]
})

with check_solutions() as check:
    check.isclose(accuracy(toy_data["true"], toy_data["pred"]), 0.6)
    check.isclose(precision(toy_data["true"], toy_data["pred"], "A"), 2/3)
    check.isclose(precision(toy_data["true"], toy_data["pred"], "C"), 0.0)
    check.isclose(recall(toy_data["true"], toy_data["pred"], "A"), 1.0)
    check.isclose(recall(toy_data["true"], toy_data["pred"], "B"), 0.5)
    check.isclose(f1_score(0, 0), 0)
    check.isclose(f1_score(0.6, 0.3), 0.4)

### Putting it all together

The function below is already complete, so you don't need to change anything here.  It puts everything together to produce an evaluation report by calling the functions you implemented above.

In [46]:
def EvaluationReport(true, pred):
    """Compute and print our own classification report."""
    acc = accuracy(true, pred)
    print(f"accuracy = {acc:.2f}")

    all_classes = sorted(set(true))
    for target in all_classes:
        p = precision(true, pred, target)
        r = recall(true, pred, target)
        f = f1_score(p, r)
        print(f"class {target:>2s}: precision = {p:.2f}, recall = {r:.2f}, f1 = {f:.2f}")

<div class="alert alert-info">
  <strong>Change the function call below</strong> so that you give it the same two arguments that you used for the classification report in Task 1!
</div>

In [47]:
# YOUR CODE HERE
# call the evaluation report
EvaluationReport(y_true, y_pred)

accuracy = 0.83
class  L: precision = 0.79, recall = 0.76, f1 = 0.78
class  R: precision = 0.85, recall = 0.87, f1 = 0.86


_Tip:_ If your implementation is correct, **you should get the exact same values** as in Task 1.

## Task 4: Compare to a baseline

Accuracy, precision, and recall should not be understood as absolute performance measures &ndash; they make sense only when we use them to compare a classifier against a **baseline**. One of the simplest possible baselines for text classification is the **most-frequent-class baseline**. This method always predicts that class which appears most often in the training data documents – without even looking at the words that appear in these documents. We would hope that a Naive Bayes classifier has higher accuracy, precision, and recall than this simple baseline.

<div class="alert alert-info">
  <strong>Write code to produce an evaluation report for the most-frequent-class baseline!</strong>

  - Use the same training and test data as before.  That means you will have to write code that inspects both the 2023/24 and 2024/25 speeches.
  - Do **not** hard-code the value of the most-frequent class; you need to derive it from the training set!
  - You can call the function `our_evaluation_report()` from Task 2 to produce the evaluation report.
</div>

In [48]:
# YOUR CODE HERE

# most freq class
mostFreqClass = speeches_202324['wing'].value_counts().idxmax()

# baseline predictions - test set 
baselinePredictions = [mostFreqClass] * len(speeches_202425)
print(f"Most frequent class in training set: {mostFreqClass}")
EvaluationReport(speeches_202425['wing'], baselinePredictions)

Most frequent class in training set: R
accuracy = 0.61
class  L: precision = 0.00, recall = 0.00, f1 = 0.00
class  R: precision = 0.61, recall = 1.00, f1 = 0.76


_Tip:_ If your implementation is correct, the accuracy should be 0.61, and the scores for class “L” (left-wing) should all be zero.

## Task 5: Reflect on the evaluation

In Task 2, we evaluated the classifier on the test data.  As a final step, let's see what happens when we evaluate the classifier on the _training data_ instead.

<div class="alert alert-info">
    <strong>Add code to evaluate the classifier on the <i>training</i> data and print a classification report!</strong>  <i>(You can copy your code from Task&nbsp;2 and just replace the test data with the training data.)</i>
</div>

In [49]:
y_true_train = speeches_202324['wing']
y_pred_train = model.predict(speeches_202324['text'])
print(classification_report(y_true_train, y_pred_train))

              precision    recall  f1-score   support

           L       0.90      0.91      0.91      5808
           R       0.94      0.94      0.94      9387

    accuracy                           0.93     15195
   macro avg       0.92      0.92      0.92     15195
weighted avg       0.93      0.93      0.93     15195



<div class="alert alert-info">
  <strong>Write a relection on how you interpret the results from the different evaluations you performed in this lab.</strong>  Use the following prompts:

  - What conclusions do you draw from the results you got in Tasks&nbsp;2 and 4?
  - How and why do the results you got here in Task&nbsp;5 differ from those in Task&nbsp;2?
  - How does this relate to general machine learning methodology?
</div>

1. Conclusions from Tasks 2 and 4: 

The trained Naive Bayes classifier performs notably better than the simple most frequent class baseline (accuracy is 0.83 vs 0.61). 

This indicates the classifier has learned useful signals from the text (it predicts the minority class “L” with non-zero precision and recall, whereas the baseline always predicts “R” and thus has zero scores for “L”). 

2. Differences between Task 5 & 2: 

Performance on the training set is higher than on the test set. 

This difference suggests the model fits the training data better than unseen data, and this can be a sign of overfitting.

3. How this relates: 

Always compare model performance to simple baselines. 

A large gap between training and test performance calls for regularisation, simpler models or more data. Reporting precision, recall and F1 score is important for imbalanced datasets to avoid misleading conclusions from accuracy alone. 💡

- - -

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

<strong>Before you submit...</strong>

➡️ Make sure you have filled in all cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.

➡️ **Make sure the notebook runs from start to finish** without errors, for example by clicking the "⏩" button in the toolbar, or via "Run$\rightarrow$Restart Kernel and Run All Cells..." in the menu.  If this is successful, you should see a green box appear below!

</div>

In [50]:
from liu_lt import success
success("Notebook runs from start to finish!")